# Edge AI Real-Time Interference Study - Colab Test & Debugging Workflow

Questo notebook permette di collaudare e fare il debugging dell'intera catena di esperimenti prima di eseguirla sulla scheda fisica (**NVIDIA Jetson AGX Orin**).

> **IMPORTANTE**: Assicurati che il runtime di Colab sia impostato con **GPU** (*Runtime > Cambia tipo di runtime > T4 GPU*).

--- 
## 1. Installazione Dipendenze di Sistema
Installiamo i tool necessari per l'ambiente Linux: `stress-ng` (generatore carichi/interferenza), `sysstat` (che fornisce `vmstat`), `cmake` e compilatore C/C++.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq stress-ng sysstat cmake build-essential openssh-server openssh-client
!pip install -q huggingface_hub datasets
!nvidia-smi

--- 
## 2. Verifica Cartella di Lavoro e File del Progetto
Se hai caricato la cartella del progetto su Google Drive o tramite `git clone`, posizionati nella root.

In [ ]:
# Se usi Google Drive scommenta le due righe sotto:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/EdgeAI_Real_Time_Interference_Performance_Study

# Verifica file presenti
!ls -la setup/

--- 
## 3. Esecuzione del Setup di Sistema (`setup_system.py`)
Scarica i dataset (**MMLU**, **Spider**), il modello **Llama-3.2-1B-Instruct** quantizzato GGUF e compila `llama.cpp` con accelerazione CUDA (`-DGGML_CUDA=ON`).

In [ ]:
!python setup/setup_system.py

--- 
## 4. (Opzionale) Test Invio e Connessione SSH

### Come testare SSH da Google Colab?
Poiché Google Colab gira su macchine virtuali cloud di Google, per testare l'invio SSH verso la tua macchina locale hai due possibilità:

1. **Mock SSH Locale (Debug immediato 100% offline su Colab)**:
   Avvia un server OpenSSH su `localhost:2222` all'interno della stessa VM di Colab con chiavi SSH autorizzate. Permette di validare che la funzione `send_files_via_ssh` di `run_experiments.py` funzioni perfettamente senza bisogno di configurazioni di rete esterne.

2. **Connessione reale verso il tuo PC di casa**:
   Poiché il tuo PC è normalmente dietro il router/NAT di casa, Colab non può raggiungere direttamente un IP privato `192.168.x.x`. Per renderlo raggiungibile puoi usare:
   - **Tailscale** (consigliato: crea una VPN privata tra il tuo PC e Colab, dandoti un IP tipo `100.x.y.z` raggiungibile da Colab).
   - Un tunnel temporaneo come **Pinggy** o **ngrok** (`ssh -R 0:localhost:22 a.pinggy.io` sul tuo PC, che ti dà un host e una porta pubblica da inserire in `--ssh-target` e `--ssh-port`).

Esegui la cella qui sotto per avviare il **Mock Server SSH** su Colab e testare subito la logica di trasferimento:

In [ ]:
# Configurazione Mock SSH Server su Colab (porta 2222)
!mkdir -p ~/.ssh /var/run/sshd
!ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N "" -q -y > /dev/null 2>&1 || ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N ""
!cat ~/.ssh/id_ed25519.pub >> ~/.ssh/authorized_keys
!chmod 600 ~/.ssh/authorized_keys ~/.ssh/id_ed25519
!chmod 700 ~/.ssh

# Genera chiavi host del server SSH se mancanti
!ssh-keygen -A > /dev/null 2>&1

# Avvia il demone sshd sulla porta 2222
!/usr/sbin/sshd -p 2222

# Smoke-test di connessione locale via SSH
import subprocess
test_res = subprocess.run(
    ["ssh", "-p", "2222", "-o", "StrictHostKeyChecking=accept-new", "-i", "/root/.ssh/id_ed25519", "root@localhost", "echo 'SSH funzionante con successo!'"],
    capture_output=True, text=True
)
print(test_res.stdout.strip() if test_res.returncode == 0 else f"Errore: {test_res.stderr}")

--- 
## 5. Test Baseline con `run_experiments.py` (senza reboot)
Eseguiamo 10 task di test della **Baseline** (senza stressor) con `vmstat` e salvataggio metriche.

> **Nota:** usiamo `--no-reboot` e `--transient-wait 0` per Colab. Includiamo `--ssh-target` puntando al server SSH (es. il mock su porta 2222 o la tua macchina).

In [ ]:
!python setup/run_experiments.py \
  --model slm \
  --device gpu \
  --benchmark mmlu \
  --no-stressor \
  --samples 10 \
  --repetitions 1 \
  --transient-wait 0 \
  --no-reboot \
  --ssh-target "root@localhost" \
  --ssh-port 2222 \
  --ssh-dest "/tmp/EdgeAI_MockRemote" \
  --ssh-key "/root/.ssh/id_ed25519"

Verifichiamo che i file CSV siano stati creati e che siano stati inviati anche nella cartella di destinazione SSH (`/tmp/EdgeAI_MockRemote`):

In [ ]:
print("=== File locali generati ===")
!ls -la data_timeseries/ results/

print("\n=== File ricevuti via SSH nella destinazione remota ===")
!ls -la /tmp/EdgeAI_MockRemote/results/ /tmp/EdgeAI_MockRemote/data_timeseries/ 2>/dev/null || echo "Nessun file ricevuto."

--- 
## 6. Test Calcolo Dimensionamento Campioni (`compute_sample_size.py`)
Calcoliamo il numero di campioni $n$ per gli stressor usando il Teorema del Limite Centrale (**CLT**), **intervallo di confidenza al 95%** ($z = 1.96$) ed **errore al 5%** sui task appena raccolti:

In [ ]:
!python setup/compute_sample_size.py \
  --model slm \
  --device gpu \
  --benchmark mmlu \
  --confidence 0.95 \
  --error 0.05

--- 
## 7. Test con Stressor (`stress-ng`)
Eseguiamo una prova di inferenza con carico di interferenza attivo (es. stressor `cpu` o `mem`):
- Verifica che `stress-ng` parta e venga terminato correttamente in sincronia con i task.
- Verifica che `vmstat` registri l'aumento di carico (es. colonne `us`, `sy`, `cs`).
- Verifica l'aggiornamento automatico della tabella aggregata del DoE.

In [ ]:
!python setup/run_experiments.py \
  --model slm \
  --device gpu \
  --benchmark mmlu \
  --use-stressor \
  --stressor cpu \
  --samples 5 \
  --repetitions 1 \
  --transient-wait 0 \
  --no-reboot \
  --ssh-target "root@localhost" \
  --ssh-port 2222 \
  --ssh-dest "/tmp/EdgeAI_MockRemote" \
  --ssh-key "/root/.ssh/id_ed25519"

--- 
## 8. Ispezione Finale dei Risultati e del DoE

In [ ]:
import pandas as pd
import glob

summary_files = glob.glob("results/doe_*.csv")
if summary_files:
    print(f"Contenuto di {summary_files[0]}:")
    display(pd.read_csv(summary_files[0]))
else:
    print("Nessun file DoE generato.")

if glob.glob("data_timeseries/*timeseries.csv"):
    ts_sample = glob.glob("data_timeseries/*timeseries.csv")[0]
    print(f"\nAnteprima Time-Series vmstat ({ts_sample}):")
    display(pd.read_csv(ts_sample).head(5))